# Vectors and rigid transforms

Points in these notebooks are row vectors with shape `(N, 3)`, and **Z is up**. A rotation `R` is active on column vectors, so `p' = R @ p + t`. For a whole cloud that is `(R @ points.T).T + t`, which is the same as `points @ R.T + t`.

State that convention before you write anything. Most bugs in this interview are a transpose, a sign, or a zero-length vector.

The solution cell replaces your function and runs the checks. Scroll past it until you have tried.


In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)


## 1. Angle and projection

Write `angles_and_projections`.

The angle between each row of `vectors` and `axis` is `arccos` of the cosine. Clamp the cosine to `[-1, 1]` before `arccos`, or floating-point error turns a parallel pair into NaN. A zero-length vector has an undefined angle; return 0 for that row.

The projection is the component of each vector along `axis`, as a vector. `axis` is not necessarily a unit vector.

Hand check: vectors `(1,0,0)`, `(0,1,0)`, `(1,1,0)` against axis `(2,0,0)` give angles `0`, `π/2`, `π/4` and projections `(1,0,0)`, `(0,0,0)`, `(1,0,0)`.

<details><summary>Hint</summary>

Normalize the axis once. Cosine is `(v · unit) / ||v||`. The projection of `v` onto `unit` is `(v · unit) * unit`.

</details>


In [ ]:
import numpy as np

def angles_and_projections(vectors, axis):
    """
    Angles of vectors with axis, and their projections onto axis.

    vectors: (N, 3)
    axis: (3,)
    returns angles (N,) in radians, projections (N, 3)
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

def angles_and_projections(vectors, axis):
    """Angles of vectors with axis, and their projections onto axis.

    vectors: (N, 3)
    axis: (3,)
    returns angles (N,) in radians, projections (N, 3)
    """
    vectors = np.asarray(vectors, dtype=float)
    axis = np.asarray(axis, dtype=float)
    axis_norm = np.linalg.norm(axis)
    if axis_norm < 1e-12:
        raise ValueError("axis has zero length")
    unit = axis / axis_norm
    vec_norm = np.linalg.norm(vectors, axis=1)
    # Zero-length vectors have an undefined angle. Return 0 and a zero projection.
    safe = np.maximum(vec_norm, 1e-12)
    cos = np.clip((vectors @ unit) / safe, -1.0, 1.0)
    angles = np.arccos(cos)
    angles = np.where(vec_norm < 1e-12, 0.0, angles)
    scalars = vectors @ unit
    projections = scalars[:, None] * unit
    return angles, projections

def check_angles_and_projections():
    vectors = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [1.0, 1.0, 0.0], [0.0, 0.0, 0.0]])
    angles, proj = angles_and_projections(vectors, np.array([2.0, 0.0, 0.0]))
    assert np.allclose(angles[:3], [0.0, np.pi / 2, np.pi / 4])
    assert angles[3] == 0.0
    assert np.allclose(proj, [[1, 0, 0], [0, 0, 0], [1, 0, 0], [0, 0, 0]])
    print("passed")

check_angles_and_projections()


## 2. Rotation about an arbitrary axis

Rotate an `(N, 3)` cloud about an axis through the origin, then add a translation.

Use Rodrigues' formula. Positive angles follow the right-hand rule: thumb along the axis, fingers curl toward the positive rotation. Return a proper rotation, so `R.T @ R = I` and `det(R) = +1`.

There are two equivalent formulas for a unit axis `k` and its skew matrix `K`:

- `R = I + sin(θ) K + (1 - cos(θ)) K²`
- `R = cos(θ) I + sin(θ) K + (1 - cos(θ)) (k kᵀ)`

`K²` and `k kᵀ` are different matrices. Mixing them is the usual bug.

Hand check: `(1,0,0)` rotated `+90°` about `+Z`, then translated by `(0,0,1)`, is `(0,1,1)`. A `+120°` rotation about `(1,1,1)` sends `(1,0,0)` to `(0,1,0)`.

<details><summary>Hint</summary>

`K = [[0, -kz, ky], [kz, 0, -kx], [-ky, kx, 0]]`. Normalize the axis before building `K`. Apply `R` to column vectors, then add `t`.

</details>


In [ ]:
import numpy as np

def rotation_matrix(axis, theta):
    """
    Active rotation by `theta` radians about `axis` (Rodrigues).

    Positive theta follows the right-hand rule. The returned matrix acts on
    column vectors: p' = R @ p.
    """
    raise NotImplementedError

def transform_points(points, axis, theta, translation):
    """
    Rotate (N, 3) points about an axis through the origin, then translate.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

def rotation_matrix(axis, theta):
    """Active rotation by `theta` radians about `axis` (Rodrigues).

    Positive theta follows the right-hand rule. The returned matrix acts on
    column vectors: p' = R @ p.
    """
    axis = np.asarray(axis, dtype=float)
    n = np.linalg.norm(axis)
    if n < 1e-12:
        raise ValueError("axis has zero length")
    k = axis / n
    K = np.array(
        [
            [0.0, -k[2], k[1]],
            [k[2], 0.0, -k[0]],
            [-k[1], k[0], 0.0],
        ]
    )
    I = np.eye(3)
    return I + np.sin(theta) * K + (1.0 - np.cos(theta)) * (K @ K)

def transform_points(points, axis, theta, translation):
    """Rotate (N, 3) points about an axis through the origin, then translate."""
    points = np.asarray(points, dtype=float)
    R = rotation_matrix(axis, theta)
    t = np.asarray(translation, dtype=float)
    return (R @ points.T).T + t

def check_transform_points():
    out = transform_points([[1.0, 0.0, 0.0]], [0.0, 0.0, 1.0], np.pi / 2, [0.0, 0.0, 1.0])
    assert np.allclose(out, [[0.0, 1.0, 1.0]], atol=1e-8)
    cycled = transform_points([[1.0, 0.0, 0.0]], [1.0, 1.0, 1.0], 2.0 * np.pi / 3, [0.0, 0.0, 0.0])
    assert np.allclose(cycled, [[0.0, 1.0, 0.0]], atol=1e-8)
    back = transform_points([[1.0, 0.0, 0.0]], [0.0, 0.0, 1.0], -np.pi / 2, [0.0, 0.0, 0.0])
    assert np.allclose(back, [[0.0, -1.0, 0.0]], atol=1e-8)
    # Rows are the images of the basis, so this matrix is R.T.
    images = transform_points(np.eye(3), [0.2, -0.4, 0.5], 0.7, [0.0, 0.0, 0.0])
    R = images.T
    assert np.allclose(R.T @ R, np.eye(3), atol=1e-8)
    assert np.allclose(np.linalg.det(R), 1.0, atol=1e-8)
    print("passed")

check_transform_points()


## 3. Compose and invert rigid transforms

A rigid transform is `p' = R @ p + t`. Write three functions:

- `apply_rigid(points, R, t)` for an `(N, 3)` cloud
- `compose_rigid(R_first, t_first, R_second, t_second)` so the result applies `first`, then `second`
- `invert_rigid(R, t)` using `R.T`, not `np.linalg.inv`

Hand check: start at `(1,0,0)`. Rotate `+90°` about Z and translate by `(1,0,0)`, then rotate `+90°` about Z again and translate by `(0,1,0)`. The point lands on `(-1, 2, 0)`. Applying the inverse returns the original point.

<details><summary>Hint</summary>

`R = R_second @ R_first` and `t = R_second @ t_first + t_second`. The inverse rotation is `R.T`, and the inverse translation is `-R.T @ t`.

</details>


In [ ]:
import numpy as np

def apply_rigid(points, R, t):
    """
    p' = R @ p + t for row points of shape (N, 3).
    """
    raise NotImplementedError

def compose_rigid(R_first, t_first, R_second, t_second):
    """
    Rigid transform that applies `first`, then `second`.
    """
    raise NotImplementedError

def invert_rigid(R, t):
    """
    Inverse of p' = R @ p + t. Uses R.T, not a general inverse.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

def apply_rigid(points, R, t):
    """p' = R @ p + t for row points of shape (N, 3)."""
    points = np.asarray(points, dtype=float)
    t = np.asarray(t, dtype=float)
    return (R @ points.T).T + t

def compose_rigid(R_first, t_first, R_second, t_second):
    """Rigid transform that applies `first`, then `second`."""
    R = R_second @ R_first
    t = R_second @ np.asarray(t_first, dtype=float) + np.asarray(t_second, dtype=float)
    return R, t

def invert_rigid(R, t):
    """Inverse of p' = R @ p + t. Uses R.T, not a general inverse."""
    t = np.asarray(t, dtype=float)
    R_inv = R.T
    return R_inv, -R_inv @ t

def check_rigid():
    def rz(angle):
        c, s = np.cos(angle), np.sin(angle)
        return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])

    R1 = rz(np.pi / 2)
    R2 = rz(np.pi / 2)
    t1 = np.array([1.0, 0.0, 0.0])
    t2 = np.array([0.0, 1.0, 0.0])
    R, t = compose_rigid(R1, t1, R2, t2)
    p = np.array([[1.0, 0.0, 0.0]])
    assert np.allclose(apply_rigid(p, R, t), [[-1.0, 2.0, 0.0]], atol=1e-8)
    R_i, t_i = invert_rigid(R, t)
    assert np.allclose(apply_rigid(apply_rigid(p, R, t), R_i, t_i), p, atol=1e-8)
    # A non-axis rotation, written out so this check does not depend on Rodrigues.
    c, s = np.cos(0.8), np.sin(0.8)
    R = np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]]) @ np.array(
        [[1.0, 0.0, 0.0], [0.0, c, -s], [0.0, s, c]]
    )
    t = np.array([1.0, 2.0, -3.0])
    pts = np.array([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.2, -0.4, 0.6]])
    R_i, t_i = invert_rigid(R, t)
    assert np.allclose(R_i @ R, np.eye(3), atol=1e-8)
    assert np.allclose(apply_rigid(apply_rigid(pts, R, t), R_i, t_i), pts, atol=1e-8)
    print("passed")

check_rigid()


## 4. A frame for a cross-section

Survey geometry is Z-up. A road tangent lives in the XY plane. Build a right-handed frame whose **columns** are forward, left, and up.

- forward: the horizontal tangent, normalized, with z = 0
- up: `(0, 0, 1)`
- left: the horizontal direction such that `forward × left = up`

A zero-length tangent should raise `ValueError`. Do not assume the input is a unit vector.

Hand check: tangent `(3, 0)` returns `I`. Tangent `(0, 1)`, looking north, has left = west = `(-1, 0, 0)`.

<details><summary>Hint</summary>

If forward is `(tx, ty, 0)`, left is `(-ty, tx, 0)`. Check the frame by `R.T @ R ≈ I` and `det(R) ≈ +1`.

</details>


In [ ]:
import numpy as np

def section_frame(tangent_xy):
    """
    Orthonormal frame whose columns are forward, left, up.

    tangent_xy: (2,) horizontal direction of travel, unnormalized.
    forward is that direction in the XY plane.
    up is +Z.
    left is horizontal so that forward x left = up (right-handed, Z-up).
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

def section_frame(tangent_xy):
    """Orthonormal frame whose columns are forward, left, up.

    tangent_xy: (2,) horizontal direction of travel, unnormalized.
    forward is that direction in the XY plane.
    up is +Z.
    left is horizontal so that forward x left = up (right-handed, Z-up).
    """
    tangent_xy = np.asarray(tangent_xy, dtype=float).reshape(2)
    n = np.linalg.norm(tangent_xy)
    if n < 1e-12:
        raise ValueError("tangent has zero horizontal length")
    forward = np.array([tangent_xy[0] / n, tangent_xy[1] / n, 0.0])
    left = np.array([-forward[1], forward[0], 0.0])
    up = np.array([0.0, 0.0, 1.0])
    return np.column_stack([forward, left, up])

def check_section_frame():
    I = section_frame(np.array([3.0, 0.0]))
    assert np.allclose(I, np.eye(3))
    north = section_frame(np.array([0.0, 2.0]))
    assert np.allclose(north[:, 0], [0, 1, 0])
    assert np.allclose(north[:, 1], [-1, 0, 0])
    assert np.allclose(north[:, 2], [0, 0, 1])
    assert np.allclose(np.linalg.det(north), 1.0)
    diag = section_frame(np.array([1.0, 1.0]))
    assert np.allclose(diag.T @ diag, np.eye(3), atol=1e-8)
    assert np.allclose(np.cross(diag[:, 0], diag[:, 1]), [0, 0, 1], atol=1e-8)
    try:
        section_frame(np.array([0.0, 0.0]))
        raise AssertionError("expected ValueError")
    except ValueError:
        pass
    print("passed")

check_section_frame()
